# Goal

從既有 ETF Trick Daily NAV/成交金額結果，建立或讀取 PIT-safe AFML dataset，並快速取得 Dollar bars、FFD、結構性統計、features、labels 與交易時點 snapshot。

## Setup

Notebook 放在 repository root。Kernel 請選擇本 repository 的 `.venv` Python。所有路徑與日期都集中在下一格，可用環境變數覆寫。

In [ ]:
import os
from pathlib import Path

from etf_tricks import ETFTrickResult
from etf_tricks.afml import AFMLConfig, ETFAFMLLab
from etf_tricks.afml import AFMLDataset

REPO_ROOT = Path.cwd().resolve()
DATA_ANALYSTS_ROOT = Path(os.getenv("DATA_ANALYSTS_ROOT", REPO_ROOT / "DataAnalysts"))
RESULT_DIR = Path(os.getenv("ETF_TRICK_RESULT_DIR", REPO_ROOT / ".artifacts/etf_tricks/performance/optimized-final-20240101-20260707"))
AFML_DATASET_DIR = Path(os.getenv("ETF_AFML_DATASET_DIR", REPO_ROOT / ".artifacts/etf_tricks/afml/optimized-final-20240101-20260707"))
TRAIN_START = os.getenv("ETF_AFML_TRAIN_START", "2024-01-01")
TRAIN_END = os.getenv("ETF_AFML_TRAIN_END", "2025-06-30")
VALIDATION_END = os.getenv("ETF_AFML_VALIDATION_END", "2025-12-31")
TEST_END = os.getenv("ETF_AFML_TEST_END", "2026-07-07")
AS_OF = os.getenv("ETF_AFML_AS_OF", TEST_END)
CONFIG = AFMLConfig()

## Build or Read

已有 AFML manifest 時直接驗 hash 後讀取；否則從既有 bounded ETFTrickResult 建立。正式測試請先使用 2024–2026，觀察數不足才延伸到 2020–2026。

In [ ]:
base_result = ETFTrickResult.read(RESULT_DIR)
if (AFML_DATASET_DIR / "manifest.json").is_file():
    dataset = AFMLDataset.read(AFML_DATASET_DIR)
else:
    lab = ETFAFMLLab.from_data_analysts(DATA_ANALYSTS_ROOT)
    dataset = lab.build_all(
        base_result,
        config=CONFIG,
        mode="train",
        train_start=TRAIN_START,
        train_end=TRAIN_END,
        validation_end=VALIDATION_END,
        test_end=TEST_END,
    )
    dataset.write(AFML_DATASET_DIR)

## Checks

這些 views 不會複製儲存資料。`for_ml` 依 split 同時檢查 feature 與 label availability；`for_trading` 絕不載入或回傳 labels。

In [ ]:
dataset.readiness
dataset.diagnostics
dataset.dollar_bars.head()
dataset.features.head()
dataset.labels.head()

In [ ]:
ml_train = dataset.for_ml("momentum", split="train")
trading_snapshot = dataset.for_trading(as_of=AS_OF, decision_cutoff="after_close")
ml_train.head()
trading_snapshot

## Next Steps

- `dataset.for_ml(etf_id, split)` 可直接交給後續 fold-local imputation、scaling、purged CV 與模型。
- `dataset.for_trading(...)` 只提供 PIT-safe feature snapshot；ETF 資金拆股仍交回既有 allocation/execution API，使用 execution-session 原始 close。
- Dollar bar 的 `bar_amount` 已是正式 feature，可用於後續 activity/threshold 研究。